# Volume Collapse Strategy - Inference Notebook

This notebook demonstrates the **Geometric Volume Collapse Strategy**, which extends RRG sector rotation with a crisis regime filter.

**Key Concept:** When cross-sectional returns collapse (all sectors move together), the geometric volume signal drops. In these crisis regimes, the strategy reduces allocation to avoid false rotation signals.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import strategy components
from strategies import VolumeCollapseStrategy, VolumeCollapseConfig
from strategies import RRGStrategy, RRGConfig
from backtesting import Backtester, WalkForwardValidator, WalkForwardConfig

## Step 1: Load Data

In [ ]:
# Load the sector price data
df = pd.read_csv("data/processed/indian_sector_data_2013_2025.csv", parse_dates=["Date"])
df = df.set_index("Date").sort_index()
df = df.apply(pd.to_numeric, errors="coerce").ffill().dropna(how="all")

# Separate benchmark and sector prices
benchmark = df["Benchmark"]
prices = df.drop(columns=["Benchmark"])

print(f"Data range: {prices.index[0]} to {prices.index[-1]}")
print(f"Sectors ({len(prices.columns)}): {list(prices.columns)}")
prices.tail()

## Step 2: Configure the Volume Collapse Strategy

The strategy combines:
- **RRG momentum signals** (RS-Ratio, RS-Momentum)
- **Geometric volume regime filter** to scale down during crises

In [ ]:
# Configure strategy
config = VolumeCollapseConfig(
    # RRG parameters (inherited)
    top_n_sectors=5,
    max_sector_weight=0.30,
    min_sector_weight=0.01,
    rs_lookback=100,
    momentum_lookback=30,
    volatility_window=46,
    use_trend_filter=True,
    trend_ma_period=50,
    weight_smoothing_alpha=0.3,
    full_allocation=True,
    rebalance_frequency="W-FRI",
    
    # Volume Collapse parameters (new)
    vol_window=60,           # Rolling window for geometric volume
    vol_percentile=0.15,     # Crisis threshold percentile
    risk_reduction_factor=0.5,  # Binary mode reduction
    min_exposure=0.2,        # Never go below 20% allocation
    smooth_scaling=True,     # Use continuous scaling
)

print("Volume Collapse Strategy Configuration:")
print(config)

In [ ]:
# Fit the strategy
strategy = VolumeCollapseStrategy(config)
strategy.fit(prices, benchmark)

print(f"Strategy fitted: {strategy.is_fitted}")

## Step 3: Inspect Geometric Volume Regime

In [ ]:
# Get volume history
vol_history = strategy.get_volume_history()

# Plot geometric volume over time
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(vol_history.index, vol_history["geometric_volume"], 
        label="Geometric Volume", linewidth=1.5, color="steelblue")
ax.plot(vol_history.index, vol_history["threshold"], 
        label="Collapse Threshold", linewidth=1, color="red", linestyle="--")

# Highlight collapse regions
collapse = vol_history["geometric_volume"] < vol_history["threshold"]
ax.fill_between(vol_history.index, 0, vol_history["geometric_volume"].max(),
                where=collapse, alpha=0.2, color="red", label="Collapse Regime")

ax.set_xlabel("Date")
ax.set_ylabel("Geometric Volume")
ax.set_title("Geometric Volume Time Series with Crisis Detection")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Get current regime
latest_date = prices.index[-1]
regime_info = strategy.get_current_regime(latest_date)

print(f"\n=== Current Regime ({regime_info['date'].date()}) ===")
print(f"Geometric Volume: {regime_info['geometric_volume']:.6f}")
print(f"Threshold:        {regime_info['threshold']:.6f}")
print(f"Regime:           {regime_info['regime']}")
print(f"Allocation Scale: {regime_info['allocation_pct']:.1f}%")

## Step 4: RRG Quadrant Classification

In [ ]:
# Get RRG quadrant classification
quadrants = strategy.get_quadrant_classification(latest_date)

print(f"RRG Classification as of {latest_date.date()}\n")
display(quadrants.sort_values("RS_Momentum", ascending=False))

In [ ]:
# Visualize RRG quadrants
fig, ax = plt.subplots(figsize=(10, 8))

colors = {
    "Leading": "green",
    "Weakening": "orange",
    "Lagging": "red",
    "Improving": "blue"
}

for sector, row in quadrants.iterrows():
    ax.scatter(row["RS_Ratio"], row["RS_Momentum"], 
               color=colors.get(row["Quadrant"], "gray"), s=100)
    ax.annotate(sector, (row["RS_Ratio"], row["RS_Momentum"]), 
                fontsize=9, ha="left")

ax.axhline(100, color="black", linewidth=0.5, linestyle="--")
ax.axvline(100, color="black", linewidth=0.5, linestyle="--")
ax.set_xlabel("RS-Ratio")
ax.set_ylabel("RS-Momentum")
ax.set_title(f"RRG Quadrant Chart - {latest_date.date()}")
ax.text(101, 101, "Leading", fontsize=10, color="green")
ax.text(101, 99, "Weakening", fontsize=10, color="orange")
ax.text(99, 99, "Lagging", fontsize=10, color="red")
ax.text(99, 101, "Improving", fontsize=10, color="blue")
plt.tight_layout()
plt.show()

## Step 5: Predict Current Weights

In [ ]:
# Get current portfolio weights
weights = strategy.predict_weights(prices, latest_date)

print(f"Portfolio Weights as of {latest_date.date()}")
print(f"Regime Scale: {regime_info['allocation_pct']:.1f}%")
print(f"Total Allocation: {weights.sum():.2%}\n")

active = weights[weights > 0].sort_values(ascending=False)
for sector, w in active.items():
    print(f"  {sector}: {w:.2%}")

## Step 6: Run Backtest - Compare RRG vs Volume Collapse

In [ ]:
# Run backtest for Volume Collapse strategy
backtester_vc = Backtester(strategy, risk_free_rate=0.05)
result_vc = backtester_vc.run(prices, benchmark)

print("=" * 50)
print("VOLUME COLLAPSE STRATEGY")
print("=" * 50)
backtester_vc.print_report(result_vc)

In [ ]:
# Run backtest for base RRG strategy (for comparison)
rrg_config = RRGConfig(
    top_n_sectors=5,
    max_sector_weight=0.30,
    min_sector_weight=0.01,
    rs_lookback=100,
    momentum_lookback=30,
    volatility_window=46,
    use_trend_filter=True,
    trend_ma_period=50,
    weight_smoothing_alpha=0.3,
    full_allocation=True,
)
rrg_strategy = RRGStrategy(rrg_config)
backtester_rrg = Backtester(rrg_strategy, risk_free_rate=0.05)
result_rrg = backtester_rrg.run(prices, benchmark)

print("=" * 50)
print("BASE RRG STRATEGY (No Volume Filter)")
print("=" * 50)
backtester_rrg.print_report(result_rrg)

In [ ]:
# Plot equity curves comparison
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(result_vc.strategy_equity, label="Volume Collapse Strategy", linewidth=2)
ax.plot(result_rrg.strategy_equity, label="Base RRG Strategy", linewidth=1.5, alpha=0.8)
ax.plot(result_vc.benchmark_equity, label="Benchmark", linewidth=1, alpha=0.6, linestyle="--")

ax.set_xlabel("Date")
ax.set_ylabel("Equity")
ax.set_title("Strategy Comparison: Volume Collapse vs Base RRG vs Benchmark")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
# Plot drawdowns comparison
def compute_drawdown(equity):
    return (equity / equity.cummax()) - 1

fig, ax = plt.subplots(figsize=(14, 4))

ax.fill_between(result_vc.strategy_equity.index, 
                compute_drawdown(result_vc.strategy_equity), 
                0, alpha=0.5, label="Volume Collapse DD", color="steelblue")
ax.fill_between(result_rrg.strategy_equity.index, 
                compute_drawdown(result_rrg.strategy_equity), 
                0, alpha=0.3, label="Base RRG DD", color="orange")

ax.set_ylabel("Drawdown")
ax.set_title("Drawdown Comparison")
ax.legend()
plt.tight_layout()
plt.show()

## Step 7: Walk-Forward Validation

In [ ]:
# Configure walk-forward validation
wf_config = WalkForwardConfig(
    train_window=504,   # ~2 years
    test_window=63,     # ~3 months
    step_size=63        # Step forward by test window
)

# Run validation for Volume Collapse
strategy_fresh = VolumeCollapseStrategy(config)
validator = WalkForwardValidator(strategy_fresh, config=wf_config, risk_free_rate=0.05)
wf_result = validator.validate(prices, benchmark)

print("=" * 50)
print("WALK-FORWARD VALIDATION (Volume Collapse)")
print("=" * 50)
validator.print_report(wf_result)

In [ ]:
# Plot OOS equity curve
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(wf_result.oos_equity, label="Out-of-Sample Equity", linewidth=2)
ax.set_xlabel("Date")
ax.set_ylabel("Equity")
ax.set_title("Walk-Forward Out-of-Sample Performance")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# View per-fold performance
wf_result.fold_metrics[["fold", "test_start", "test_end", "total_return", "max_drawdown"]]

---

## Summary

The **Volume Collapse Strategy** extends the RRG approach with:

1. **Geometric Volume** - Measures cross-sectional return dimensionality
2. **Regime Detection** - Identifies crisis periods when rotation signals are unreliable
3. **Dynamic Scaling** - Reduces allocation during collapses, preserves returns in normal regimes

Compare the metrics above to assess if the volume filter improves risk-adjusted returns!